# Boundary Conditions for DG

This is a simple case to develop and test my boundary conditions framework. The advection equation is solved on a square with different conditions at inflows.

$$
\begin{cases}
    \frac{\mathrm{d} \iota}{\mathrm{d} t} + \nabla \cdot (\iota \mathbf{v}) = 0 &x \in \Omega \\
    \iota = \iota_{1} &x \in \Gamma_{1} \cap \Gamma_{\mathrm{in}} \\
    \iota = \iota_{2}(t) &x \in \Gamma_{2} \cap \Gamma_{\mathrm{in}}
\end{cases}
$$

## Imports and Setup

In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
import dolfinx as dx
import dolfinx.plot as dxp
import fenicsxtools as ft
from mpi4py import MPI
import numpy as np
from petsc4py import PETSc
import pyvista as pv
import ufl

In [23]:
pv.set_jupyter_backend('static')
plotter = pv.Plotter()

## Problem Parameters

Here, we give more specifics for the problem.

$$
\begin{align}
\mathbf{v} &= v_{0} \mathbf{\hat{e}}_{x} \\
\iota_{0}(\mathbf{x}) &= 0 \\
\iota(\mathbf{x}, t) &= e^{-\frac{|\mathbf{x} - \mathbf{x}_{0} - \mathbf{v} t|^{2}}{2 \sigma^{2}}}
\end{align}
$$

In [24]:
# Equation parameters
h0 = 1.0 # Constant height
u0 = 1.0 # Constant rightward flow
v0 = 0.0 # No vertical flow
d0 = 0.0 # No diffusion

In [25]:
# Initial condition
x0 = 0.5
y0 = 0.5
r0 = 0.1
def iota0(x):
    return np.zeros(x[0].shape)

## Discretization and Meshing Parameters

A simple unit square $\Omega = (0, 1)^{2}$ is used for the domain.

In [26]:
# Numerical parameters
nx = 10 # Cell resolution
dt = 0.001 # Time step
nt = 2000
ntps = 1 / dt
write_every = 10 # Only every nth frame
plot_speedup = 1 # Relative to sim time

In [27]:
# Define a simple square
domain = dx.mesh.create_unit_square(MPI.COMM_WORLD, nx, nx, cell_type=dx.mesh.CellType.triangle)

In [28]:
# Define boundaries for applying boundary conditions
# Values are defined after function space construction
def is_gamma_1_in(x):
    return (
        np.isclose(x[0], 0.0)
        & (x[1] >= 0.5) # Note equality for full inclusion
    )
GAMMA_1_IN_MARKER = 1 # Do not use tag 0, the default for "otherwise"
def is_gamma_2_in(x):
    return (
        np.isclose(x[0], 0.0)
        & (x[1] <= 0.5) # Note equality for full inclusion
    )
GAMMA_2_IN_MARKER = 2

In [29]:
# Construct measure from boundary locations
gamma_1_in_facets = dx.mesh.locate_entities_boundary(
    msh=domain,
    dim=domain.topology.dim - 1,
    marker=is_gamma_1_in
)
gamma_2_in_facets = dx.mesh.locate_entities_boundary(
    msh=domain,
    dim=domain.topology.dim - 1,
    marker=is_gamma_2_in
)
boundaries = np.hstack([
    gamma_1_in_facets,
    gamma_2_in_facets
]).astype(np.int32)
markers = np.hstack([
    np.full_like(gamma_1_in_facets, GAMMA_1_IN_MARKER),
    np.full_like(gamma_2_in_facets, GAMMA_2_IN_MARKER)
]).astype(np.int32)
sorted_indices = np.argsort(boundaries)
facet_tags = dx.mesh.meshtags(
    msh=domain,
    dim=domain.topology.dim - 1,
    entities=boundaries[sorted_indices],
    values=markers[sorted_indices]
)
ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)

## FEM Formulation

In [30]:
# First, define equation
V = dx.fem.functionspace(domain, ('Discontinuous Lagrange', 1))
iota = dx.fem.Function(V, name='iota') # The conserved quantity
iota.interpolate(iota0)
equation = ft.equations.get_constant_advection(
    domain=domain,
    U=iota,
    v=(u0, v0)
)

In [31]:
# Next, define boundary conditions using mesh and function space
time = dx.fem.Constant(domain, 0.0)
iota_1_in = dx.fem.Constant(domain, 1.0)
iota_2_in = ufl.max_value(
    dx.fem.Constant(domain, 1.0) - time,
    dx.fem.Constant(domain, 0.0)
)
def update_iota_2_in(tt):
    global time
    time.value = tt
boundary_conditions = [
    ft.bcs.DirichletCondition(GAMMA_1_IN_MARKER, iota_1_in),
    ft.bcs.DirichletCondition(GAMMA_2_IN_MARKER, iota_2_in, update=update_iota_2_in)
]

In [32]:
system = ft.formulations.SemiDiscreteSystem.using_DG(equation, ft.fluxes.llf, boundary_conditions=boundary_conditions, ds=ds)
ts = ft.solvers.ts_from_SemiDiscreteSystem(system, 'explicit')

In [33]:
# Set time stepping scheme
ts.setType(PETSc.TS.Type.EULER)
ts.setTime(0.0)
ts.setTimeStep(dt)
ts.setMaxTime(dt * nt)
ts.setExactFinalTime(PETSc.TS.ExactFinalTime.MATCHSTEP)

In [34]:
# Set up plotting preliminaries
topology, cell_types, geometry = dxp.vtk_mesh(V)
grid = pv.UnstructuredGrid(topology, cell_types, geometry)

In [35]:
# Numerical plotter
grid.point_data['iota_numerical'] = np.asarray(iota.x.array.real).copy()
plotter_numerical = pv.Plotter()
plotter_numerical.add_mesh(
    grid,
    scalars='iota_numerical',
    cmap='jet',
    clim=[0.0, 1.0],
    show_edges=False
)
plotter_numerical.view_xy()
plotter_numerical.open_gif('drifting_gaussian_numerical.gif', fps=(plot_speedup * ntps / write_every))

In [36]:
# Set monitor for time stepping loop
def monitor(ts, step, t, u):
    if step % write_every != 0:
        return
    # Make sure solution is updated
    dx.fem.petsc.assign(u, iota)
    iota.x.scatter_forward()
    # iota_true.interpolate(lambda x : iota_true_function(x, t))
    # Plotting from rank 0
    if domain.comm.rank == 0:
        grid.point_data['iota_numerical'] = np.asarray(iota.x.array.real).copy()
        # grid.point_data['iota_true'] = np.asarray(iota_true.x.array.real).copy()
        # grid.point_data['iota_error'] = np.asarray(iota.x.array.real - iota_true.x.array.real).copy()
        plotter_numerical.write_frame()
        # plotter_true.write_frame()
        # plotter_error.write_frame()
ts.setMonitor(monitor)

In [37]:
# Finalize setup
ts.setFromOptions()

## Solution Loop

In [38]:
# Solution loop
ts.solve(iota.x.petsc_vec)

In [39]:
# Close and save file
plotter_numerical.close()
# plotter_true.close()
# plotter_error.close()